# LifeMate — Fine-tune có tốt hơn chưa? :))

**Add Input output của hai notebook train → điền đường dẫn ở cell 1 → CPU → Run All.**
Không cần GPU, Internet, tải checkpoint hoặc cài thư viện. Notebook chỉ đọc JSON/JSONL.
Nếu điền thư mục mount ngoài cùng, notebook tự tìm các run con có `baseline_metrics.json`.

Bạn sẽ thấy: kết luận so với pretrained baseline của từng model; EM/WER/CER/SER từng epoch;
biểu đồ; điểm theo nhóm; câu sửa đúng và câu sai thêm; trạng thái train và file còn thiếu.
EM càng cao càng tốt; WER/CER/SER càng thấp càng tốt. WER không phải accuracy.

**Đây là validation dùng chọn checkpoint, chưa chứng minh tốt hơn trên test độc lập.**
Không so với benchmark 991 cũ. English-term là nhãn gợi ý lexicon, không phải nhãn giọng/code-switch thật.
Best được suy ra từ các lần đánh giá hợp lệ theo EM cao nhất, hòa lấy WER thấp nhất;
không đọc `best.pt`, nên không xác nhận trực tiếp checkpoint đó chứa weights nào.


In [ ]:
from pathlib import Path

INPUT_DIRS = {
    "hynt": "",       # Dán đường dẫn output notebook thứ nhất
    "sherpa_vi": "",  # Dán đường dẫn output notebook thứ hai
}
MAX_EXAMPLES = 12  # Số câu hiển thị cho mỗi nhóm lỗi của mỗi run
REPORT_DIR = Path("/kaggle/working/finetune_review")
# Chỉ xem một run: xóa dòng input còn lại. Có thể thêm run 1 epoch/5 epoch để xem chung.


In [ ]:
import base64
import csv
import html
import io
import json
import math
import unicodedata
from IPython.display import HTML, FileLink, display

FRAGMENTS = []
SUMMARY = []
REVIEWS = []
def show(markup):
    FRAGMENTS.append(markup)
    display(HTML(markup))

def esc(value):
    return html.escape(str(value))

def table(rows):
    if not rows:
        show('<p>Không có.</p>')
        return
    keys = list(rows[0])
    show('<div style="overflow:auto"><table border="1" cellpadding="6" style="border-collapse:collapse">'
         + '<tr>' + ''.join('<th>'+esc(k)+'</th>' for k in keys) + '</tr>'
         + ''.join('<tr>'+''.join('<td>'+esc(row.get(k, ''))+'</td>' for k in keys)+'</tr>' for row in rows)
         + '</table></div>')

def read_json(path):
    return json.loads(path.read_text(encoding='utf-8'))

def records(path):
    if not path.is_file():
        raise ValueError('Thiếu '+path.name)
    result = {}
    for line in path.read_text(encoding='utf-8').splitlines():
        if not line.strip():
            continue
        row = json.loads(line)
        uid = row['uid']
        if uid in result:
            raise ValueError(f'{path.name}: trùng uid {uid}')
        result[uid] = row
    return result

def normalize(text):
    text = unicodedata.normalize('NFC', str(text)).lower()
    return ' '.join(''.join(' ' if unicodedata.category(c)[0] in 'PS' else c for c in text).split())

def distance(a, b):
    prev = list(range(len(b)+1))
    for i, x in enumerate(a, 1):
        cur = [i]
        for j, y in enumerate(b, 1):
            cur.append(min(cur[-1]+1, prev[j]+1, prev[j-1]+(x != y)))
        prev = cur
    return prev[-1]

def score(rows):
    n = len(rows)
    words = sum(r['n_words'] for r in rows)
    chars = sum(r['n_chars'] for r in rows)
    exact = sum(r['exact'] for r in rows)
    return dict(n=n, exact=exact, EM=100*exact/n if n else 0,
                WER=100*sum(r['word_edits'] for r in rows)/words if words else 0,
                CER=100*sum(r['char_edits'] for r in rows)/chars if chars else 0,
                SER=100*(1-exact/n) if n else 0)

def measured(metrics_path, valid, fingerprint):
    data = read_json(metrics_path)
    if not fingerprint or data.get('split_fingerprint') != fingerprint:
        raise ValueError('Thiếu hoặc khác split_fingerprint')
    pred = records(metrics_path.with_name(metrics_path.name.replace('_metrics.json', '_predictions.jsonl')))
    if set(pred) != set(valid):
        raise ValueError(f'UID khác valid: thiếu {len(set(valid)-set(pred))}, thừa {len(set(pred)-set(valid))}')
    if not valid:
        raise ValueError('valid rỗng')
    rows = {}
    for uid, gold in valid.items():
        p = pred[uid]
        if p.get('status') != 'ok' or not isinstance(p.get('hypothesis'), str):
            raise ValueError(f'Prediction lỗi/thiếu hypothesis: {uid}')
        ref, hyp = normalize(gold['reference']), normalize(p['hypothesis'])
        if normalize(p['reference']) != ref:
            raise ValueError(f'Reference khác valid: {uid}')
        rows[uid] = dict(uid=uid, reference=gold['reference'], hypothesis=p['hypothesis'],
                         exact=int(ref == hyp), word_edits=distance(ref.split(), hyp.split()),
                         char_edits=distance(ref.replace(' ', ''), hyp.replace(' ', '')),
                         n_words=len(ref.split()), n_chars=len(ref.replace(' ', '')))
    s = score(list(rows.values()))
    saved = data['buckets']['all']
    expected = dict(n=s['n'], n_ok=s['n'], n_error=0, exact_normalized=s['exact'],
                    word_edits=sum(r['word_edits'] for r in rows.values()),
                    char_edits=sum(r['char_edits'] for r in rows.values()),
                    n_words=sum(r['n_words'] for r in rows.values()),
                    n_chars=sum(r['n_chars'] for r in rows.values()))
    for key, value in expected.items():
        if saved.get(key) != value:
            raise ValueError(f'Metrics không khớp predictions: {key}={saved.get(key)} thay vì {value}')
    for key in ('wer', 'cer'):
        if not math.isclose(saved[key]*100, s[key.upper()], abs_tol=1e-7):
            raise ValueError(f'{key} không khớp predictions')
    return dict(name=metrics_path.name.removesuffix('_metrics.json'),
                epoch=data.get('epoch', 0), step=data.get('global_step', 0), rows=rows, **s)

def metric_row(item):
    return {'Lần đánh giá': item['name'], 'Epoch': item['epoch'], 'Update': item['step'],
            'Đúng / tổng': f"{item['exact']}/{item['n']}",
            **{k+' %': round(item[k], 3) for k in ('EM', 'WER', 'CER', 'SER')}}

def review(label, root):
    show('<hr><h2>'+esc(label)+'</h2><p>'+esc(root)+'</p>')
    split = read_json(root/'split_summary.json')
    valid = records(root/'valid.jsonl')
    fingerprint = split.get('split_fingerprint')
    if len(valid) != split.get('valid_audio'):
        raise ValueError('valid.jsonl không đủ valid_audio trong split_summary')
    state = read_json(root/'state.json') if (root/'state.json').is_file() else {}
    show('<p><b>Trạng thái:</b> '+esc(state.get('state', 'thiếu state.json'))
         +' | updates: '+esc(state.get('updates', '?'))
         +' | valid: '+str(len(valid))+' câu | split: '+esc(fingerprint)+'</p>')
    table([{'File': name, 'Có trong input': (root/name).is_file()} for name in
           ('best.pt', 'last.pt', 'bpe.model', 'train_config.json', 'run_identity.json', 'failure.json')])
    if (root/'failure.json').exists():
        show('<details><summary>failure.json — cần đọc cùng trạng thái mới nhất</summary><pre>'
             +esc((root/'failure.json').read_text(encoding='utf-8'))+'</pre></details>')
    show('<p>Conversation-disjoint: '+esc(split.get('conversation_disjoint'))
         +' | PCM-disjoint: '+esc(split.get('pcm_disjoint'))
         +' | Speaker-disjoint verified: '+esc(split.get('speaker_disjoint_verified'))
         +'. Điểm dưới đây là validation; chưa phải test độc lập.</p>')
    evaluations, errors = [], []
    for path in sorted(root.glob('*_metrics.json')):
        name = path.name.removesuffix('_metrics.json')
        if not (name == 'baseline' or name.startswith(('epoch_', 'partial_epoch_'))):
            continue  # train_mining is training data, never a validation candidate
        try:
            evaluations.append(measured(path, valid, fingerprint))
        except Exception as exc:
            errors.append({'File bị loại khỏi so sánh': path.name, 'Lý do': str(exc)})
    if errors:
        table(errors)
    evaluations.sort(key=lambda x: (x['step'], x['epoch'], x['name']))
    show('<h3>Điểm từng lần đánh giá — tính lại từ predictions</h3>')
    table([metric_row(x) for x in evaluations])
    baseline = next((x for x in evaluations if x['name'] == 'baseline'), None)
    trained = [x for x in evaluations if x['name'] != 'baseline' and x['step'] > 0]
    if baseline is None or not trained:
        show('<p><b>Chưa đủ bằng chứng để kết luận:</b> cần baseline và ít nhất một lần validation sau optimizer update, cùng đủ predictions.</p>')
        return
    # Stable order preserves the first candidate on an exact tie, as the trainer does.
    best = max([baseline]+trained, key=lambda x: (x['EM'], -x['WER']))
    best_trained = max(trained, key=lambda x: (x['EM'], -x['WER']))
    latest = max(trained, key=lambda x: (x['step'], x['epoch']))
    delta = {k: best_trained[k]-baseline[k] for k in ('EM', 'WER', 'CER', 'SER')}
    verdict = ('Có cải thiện trên validation theo tiêu chí EM → WER.' if best is not baseline
               else 'Chưa vượt baseline theo tiêu chí EM → WER; baseline vẫn là lựa chọn tốt nhất trong các kết quả đọc được.')
    show('<h3>'+esc(verdict)+'</h3><p>Best suy ra (kể cả baseline): <b>'+esc(best['name'])
         +'</b>. Best sau train: <b>'+esc(best_trained['name'])+'</b>. Lần đánh giá cuối: '+esc(latest['name'])+'.</p>')
    table([{'So sánh': 'Best sau train − baseline (điểm phần trăm)',
            **{k: f'{delta[k]:+.3f}' for k in delta},
            'Số câu đúng tăng': best_trained['exact']-baseline['exact']}])
    show('<p>Δ EM dương là tốt; Δ WER/CER/SER âm là tốt. Nếu các metric đi ngược chiều, cần cân nhắc đánh đổi. '
         'Chọn best trên chính valid có thể làm kết quả lạc quan; cần test mới để xác nhận khả năng tổng quát.</p>')
    if errors:
        show('<p><b>Có lần đánh giá bị loại:</b> best chỉ dựa trên các file hợp lệ còn lại; chưa khẳng định best của toàn bộ quá trình train.</p>')
    try:
        import matplotlib.pyplot as plt
        fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
        names = [x['name'] for x in evaluations]
        for ax, keys in zip(axes, [('EM',), ('WER', 'CER')]):
            for key in keys:
                ax.plot(range(len(names)), [x[key] for x in evaluations], marker='o', label=key)
            ax.set_xticks(range(len(names)), names, rotation=45, ha='right')
            ax.set_ylabel('%'); ax.grid(alpha=.25); ax.legend()
        fig.suptitle(label+' — validation'); fig.tight_layout()
        buffer = io.BytesIO(); fig.savefig(buffer, format='png', dpi=130, bbox_inches='tight'); plt.close(fig)
        show('<img style="max-width:100%" src="data:image/png;base64,'+base64.b64encode(buffer.getvalue()).decode()+'">')
    except ImportError:
        show('<p>Thiếu matplotlib; xem bảng điểm thay biểu đồ.</p>')
    groups = {'all': list(valid),
              'english_term_hint': [u for u, r in valid.items() if r.get('english_term_hint')],
              'contains_digits': [u for u, r in valid.items() if any(c.isdigit() for c in r['reference'])]}
    for kind in sorted({r.get('kind', '') for r in valid.values()} - {''}):
        groups['kind: '+kind] = [u for u, r in valid.items() if r.get('kind') == kind]
    bucket_rows = []
    for group, uids in groups.items():
        if not uids:
            continue
        a = score([baseline['rows'][u] for u in uids])
        b = score([best_trained['rows'][u] for u in uids])
        bucket_rows.append({'Nhóm (có thể giao nhau)': group, 'N': len(uids),
                            'EM trước %': round(a['EM'], 3), 'EM sau %': round(b['EM'], 3),
                            'WER trước %': round(a['WER'], 3), 'WER sau %': round(b['WER'], 3),
                            'CER trước %': round(a['CER'], 3), 'CER sau %': round(b['CER'], 3)})
    show('<h3>Theo nhóm — baseline so với best sau train</h3>'); table(bucket_rows)
    pairs = [(baseline['rows'][u], best_trained['rows'][u]) for u in valid]
    fixed = [(a,b) for a,b in pairs if not a['exact'] and b['exact']]
    broken = [(a,b) for a,b in pairs if a['exact'] and not b['exact']]
    still_wrong = sorted([(a,b) for a,b in pairs if not b['exact']],
                         key=lambda pair: pair[1]['word_edits'], reverse=True)
    show(f'<h3>Câu cụ thể</h3><p>Sai → đúng: <b>{len(fixed)}</b> | đúng → sai: <b>{len(broken)}</b> '
         f'| đúng ở cả hai: {sum(a["exact"] and b["exact"] for a,b in pairs)} '
         f'| sai ở cả hai: {sum(not a["exact"] and not b["exact"] for a,b in pairs)}. '
         '“Đúng” ở đây là trùng transcript sau chuẩn hóa, chưa phải đánh giá ngữ nghĩa bằng người.</p>')
    for title, examples in [('Sửa được', fixed), ('Sai thêm', broken), ('Còn sai — nhiều word edits nhất', still_wrong)]:
        show('<h4>'+title+f' (hiện tối đa {MAX_EXAMPLES})</h4>')
        table([{'UID': a['uid'], 'Reference': a['reference'], 'Baseline': a['hypothesis'],
                'Sau train': b['hypothesis'], 'Word edits trước → sau': f'{a["word_edits"]} → {b["word_edits"]}'}
               for a,b in examples[:MAX_EXAMPLES]])
    summary = {'Run': label, 'Best suy ra': best['name'], 'Best sau train': best_trained['name'],
               'EM baseline %': round(baseline['EM'], 3), 'EM best sau train %': round(best_trained['EM'], 3),
               'WER baseline %': round(baseline['WER'], 3), 'WER best sau train %': round(best_trained['WER'], 3),
               'CER best sau train %': round(best_trained['CER'], 3), 'Sửa được': len(fixed), 'Sai thêm': len(broken),
               'State': state.get('state', '?'), 'Lần đánh giá bị loại': len(errors)}
    SUMMARY.append(summary)
    REVIEWS.append(dict(label=label, fingerprint=fingerprint, valid=valid, best=best))

if not INPUT_DIRS or any(not str(p).strip() for p in INPUT_DIRS.values()):
    raise ValueError('Điền các đường dẫn INPUT_DIRS trước; xóa dòng nếu không dùng run đó.')
if not isinstance(MAX_EXAMPLES, int) or MAX_EXAMPLES < 1:
    raise ValueError('MAX_EXAMPLES phải là số nguyên dương')
show('<h1>LifeMate — báo cáo fine-tune</h1>')
seen = set()
for label, raw in INPUT_DIRS.items():
    root = Path(raw).expanduser().resolve(strict=True)
    candidates = [root] if (root/'baseline_metrics.json').is_file() else sorted({p.parent for p in root.rglob('baseline_metrics.json')})
    if not candidates:
        show('<p><b>'+esc(label)+': không thấy baseline_metrics.json. Chọn output đã Save Version của notebook train.</b></p>')
    for candidate in candidates:
        if candidate in seen:
            continue
        seen.add(candidate)
        run_label = label if len(candidates) == 1 else label+' / '+str(candidate.relative_to(root))
        try:
            review(run_label, candidate)
        except Exception as exc:
            show('<p><b>Không kết luận cho '+esc(run_label)+': '+esc(exc)+'</b></p>')

show('<hr><h2>Tổng hợp các run</h2>'); table(SUMMARY)
if len(REVIEWS) > 1:
    first = REVIEWS[0]
    def same_valid(item):
        return (item['fingerprint'] == first['fingerprint'] and set(item['valid']) == set(first['valid'])
                and all(normalize(item['valid'][u]['reference']) == normalize(first['valid'][u]['reference']) for u in first['valid']))
    if all(same_valid(item) for item in REVIEWS):
        show('<p>Các run có cùng fingerprint, UID và reference valid. Có thể đối chiếu điểm trên tập này; '
             'điều đó chưa thay thế test độc lập. Bảng sau dùng best suy ra của mỗi run, có thể là baseline.</p>')
        table([{'Run': item['label'], **metric_row(item['best'])} for item in REVIEWS])
    else:
        show('<p><b>Các run khác split/UID/reference: không xếp hạng model trực tiếp từ các điểm này.</b></p>')

REPORT_DIR.mkdir(parents=True, exist_ok=True)
report = REPORT_DIR/'finetune_review.html'
report.write_text('<!doctype html><html><head><meta charset="utf-8"><title>LifeMate fine-tune review</title>'
                  '<style>body{font:15px system-ui;max-width:1250px;margin:30px auto;padding:15px}td{vertical-align:top;white-space:normal}th{background:#eef2f7}pre{white-space:pre-wrap}</style>'
                  '</head><body>'+''.join(FRAGMENTS)+'</body></html>', encoding='utf-8')
if SUMMARY:
    with (REPORT_DIR/'summary.csv').open('w', encoding='utf-8-sig', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=list(SUMMARY[0])); writer.writeheader(); writer.writerows(SUMMARY)
print('Báo cáo nhẹ (không có checkpoint):', report)
import os
display(FileLink(os.path.relpath(report, Path.cwd())))
